## 01.Dynamic Views for Data Level Security — objective

- Compare raw fields with typed records and parsing failures.
- Explain how identity and governance affect visible data and permitted actions.

Historical results; no code executed.

# 01.Dynamic Views for Data Level Security

Read the original source and historical results; nothing was executed.

[Full original payload](../../../artifacts/notebook-payloads/dea09-data-security/01.Dynamic%20Views%20for%20Data%20Level%20Security.json)

# Data-Level Security Using Dynamic Views

This notebook demonstrates how data-level security can be implemented using dynamic views.

The implementation covers two key capabilities:

- Row-Level Security: Controls which rows are visible to a user  
- Column-Level Masking: Controls how sensitive values are displayed  

A simple sales dataset is used to illustrate how different users receive different results when querying the same view.

#### Step 1: Set up the schema for the table

Archived Databricks source — shown for reading, not Python execution.

```sql
USE CATALOG demo;
CREATE SCHEMA IF NOT EXISTS data_security;
```

In [ ]:
# Archived output only; no computation.

Archived rich output omitted. Full archived values are in the linked payload reference in the notebook header.

Archived Databricks source — shown for reading, not Python execution.

```sql
USE SCHEMA data_security;
```

In [ ]:
# Archived output only; no computation.

Archived rich output omitted. Full archived values are in the linked payload reference in the notebook header.

#### Step 2: Create the table
A table is defined to represent sales data across multiple regions.

Archived Databricks source — shown for reading, not Python execution.

```sql
CREATE OR REPLACE TABLE sales_dynamic (
  id INT,
  region STRING,
  email STRING,
  revenue INT
);
```

In [ ]:
# Archived output only; no computation.

Archived rich output omitted. Full archived values are in the linked payload reference in the notebook header.

#### Step 3: Insert sample data
The dataset includes records from both UK and US regions.

This enables validation of row-level security by ensuring that different users see only relevant regional data.

Archived Databricks source — shown for reading, not Python execution.

```sql
INSERT INTO sales_dynamic VALUES
  (1, 'UK', 'john.doe@email.com', 10000),
  (2, 'US', 'mike.smith@email.com', 20000),
  (3, 'UK', 'sara.jones@email.com', 15000),
  (4, 'US', 'david.brown@email.com', 25000),
  (5, 'UK', 'emma.wilson@email.com', 18000);
```

In [ ]:
# Archived output only; no computation.

num_affected_rows,num_inserted_rows
5,5


Archived Databricks source — shown for reading, not Python execution.

```sql
SELECT * FROM sales_dynamic;
```

In [ ]:
# Archived output only; no computation.

id,region,email,revenue
1,UK,john.doe@email.com,10000
2,US,mike.smith@email.com,20000
3,UK,sara.jones@email.com,15000
4,US,david.brown@email.com,25000
5,UK,emma.wilson@email.com,18000


#### Step 4: Create the dynamic view
The dynamic view implements both row-level security and column-level masking.

##### Row-Level Security

- UK group → returns only rows where region = 'UK'  
- US group → returns only rows where region = 'US'  
- Admin group → returns all rows  

##### Column-Level Masking

- Admin group → full email visible  
- Other users → email is masked  

##### Key Function: is_account_group_member()

This function evaluates whether the current user belongs to a specified group.

Examples:

- is_account_group_member('uk-sg') 
- is_account_group_member('us-sg') 
- is_account_group_member('admin-sg')  

The function returns TRUE or FALSE and is used to drive conditional logic inside the view.

Archived Databricks source — shown for reading, not Python execution.

```sql
SELECT is_account_group_member('admin-sg'), is_account_group_member('uk-sg')
```

In [ ]:
# Archived output only; no computation.

is_account_group_member(admin-sg),is_account_group_member(uk-sg)
False,True


Archived Databricks source — shown for reading, not Python execution.

```sql
CREATE OR REPLACE VIEW vw_sales_dynamic AS
SELECT id,
       region,
       CASE 
         WHEN is_account_group_member('admin-sg') THEN email
         ELSE concat(substr(email, 1, 1), '***@', split(email, '@')[1])
       END AS email,  
       revenue
FROM sales_dynamic
WHERE is_account_group_member('admin-sg')
OR (is_account_group_member('uk-sg') AND region = 'UK')
OR (is_account_group_member('us-sg') AND region = 'US');
```

In [ ]:
# Archived output only; no computation.

Archived rich output omitted. Full archived values are in the linked payload reference in the notebook header.

#### Step 5: Query the dynamic view

Archived Databricks source — shown for reading, not Python execution.

```sql
SELECT * FROM vw_sales_dynamic;
```

In [ ]:
# Archived output only; no computation.

id,region,email,revenue
1,UK,j***@email.com,10000
3,UK,s***@email.com,15000
5,UK,e***@email.com,18000
